<h1> MAC0508 EP2 - Tradução Automática de Baixo Recurso </h1> 
<h3> Vitor Voigt Gava (NUSP: 14611842) </h3>


<h4> Baseado em MBart50.ipynb (disponibilizado no E-Disciplinas) 
<br> Adaptado para modelo NLLB </h4>

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import ipywidgets as widgets
from IPython.display import display
from tqdm import tqdm
import unicodedata
import pandas as pd



In [2]:
MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
PORT = "por_Latn"   # Português
TUPI = "grn_Latn"   # “Tupi Antigo” ≈ Guarani
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

# Pré-processamento + Carregamento

Partição do Corpus em Treino (70%), Validação (15%), Teste (15%)

In [ ]:
from sklearn.model_selection import train_test_split

# Carregar a planilha
file_path = "Cópia de portugues-guarani-tupi antigo.xlsx"
df = pd.read_excel(file_path)

# Padronizar os nomes das colunas
df.columns = ["portugues", "tupi"]

# Remover linhas vazias ou com NaN
df = df.dropna()

# Embaralhar o dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Divisão: 70% treino, 15% validação, 15% teste
train, temp = train_test_split(
    df, test_size=0.30, random_state=42
)

validation, test = train_test_split(
    temp, test_size=0.50, random_state=42
)

# Salvar os arquivos
train.to_csv("train.csv", index=False)
validation.to_csv("validation.csv", index=False)
test.to_csv("test.csv", index=False)

print("Arquivos gerados:")
print("train.csv")
print("validation.csv")
print("test.csv")

print("\nTamanhos:")
print("Treino:", len(train))
print("Validação:", len(validation))
print("Teste:", len(test))

Ajeitar as linhas do CSV para o formato "frase em portugues","frase em tupi"

In [18]:
import re

def fix_line(line):
    line = line.strip()

    # Encontrar primeira vírgula NÃO dentro de aspas
    inside_quotes = False
    split_index = None

    for i, ch in enumerate(line):
        if ch == '"':
            inside_quotes = not inside_quotes
        elif ch == ',' and not inside_quotes:
            split_index = i
            break

    if split_index is None:
        return line  # linha inválida, devolve como está

    pt = line[:split_index].strip()
    tupi = line[split_index + 1:].strip()

    # Remover aspas externas se existirem
    if pt.startswith('"') and pt.endswith('"'):
        pt = pt[1:-1]
    if tupi.startswith('"') and tupi.endswith('"'):
        tupi = tupi[1:-1]

    # Recolocar aspas corretamente nos dois lados
    pt = f'"{pt}"'
    tupi = f'"{tupi}"'

    return f"{pt},{tupi}"


def fix_csv(path_in, path_out):
    out_lines = []

    with open(path_in, "r", encoding="utf-8") as f:
        for line in f:
            fixed = fix_line(line)
            out_lines.append(fixed)

    with open(path_out, "w", encoding="utf-8") as f:
        for l in out_lines:
            f.write(l + "\n")

    print(f"Arquivo corrigido salvo em {path_out}")


Limpeza das linhas

In [22]:
import re
import unicodedata

def normalize_apostrophes(t):
    if not isinstance(t, str):
        return t
    return (
        t.replace("’", "'")
         .replace("‘", "'")
         .replace("ˈ", "'")
    )

def clean_pred_pt(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)

    # remover aspas no início/fim
    text = re.sub(r'^["“”\'`]+', "", text)
    text = re.sub(r'["“”\'`]+$', "", text)

    text = normalize_apostrophes(text)

    # remover espaços múltiplos
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_pred_tupi(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)

    # remover aspas no início/fim
    text = re.sub(r'^["“”\'`]+', "", text)
    text = re.sub(r'["“”\'`]+$', "", text)

    text = normalize_apostrophes(text)

    # trocar ñ → nh
    text = text.replace("ñ", "nh")

    # remover espaços múltiplos
    text = re.sub(r"\s+", " ", text).strip()

    # padronizar minúsculas
    text = text.lower()

    return text


Carregamento do Dataset

In [3]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": "csv's/train_fixed.csv",
        "validation": "csv's/validation_fixed.csv",
        "test": "csv's/test_fixed.csv"
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Função auxiliar para a tradução de sentenças 
(Empregada em ambos regimes: Zero-Shot e Few-Shot)

In [14]:
def traduzir(frase, src_lang, tgt_lang, tokenizer, model):

    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang

    bos_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    encoded = tokenizer(frase, return_tensors="pt").to(model.device)

    generated = model.generate(
        **encoded,
        forced_bos_token_id=bos_id,
        max_length=128
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)


# Zero-Shot


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_CHECKPOINT,
    device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    src_lang=PORT,
    tgt_lang=TUPI
)

<h3> Tradução do Corpus Teste </h3>

In [ ]:
# DEMORA 35 MINUTOS PARA RODAR
from tqdm import tqdm

test_pt = dataset["test"]["portugues"]     # frases de entrada em pt
test_tupi = dataset["test"]["tupi"]        # frases de entrada em tupi
translations2tupi = []                     # armazenará as traduções em tupi
translations2pt = []                       # armazenará as traduções em pt

# PT -> TUPI
for sentence in tqdm(test_pt, desc="PT->Tupi"):
    translations2tupi.append(traduzir(sentence, PORT, TUPI, tokenizer, model))

# TUPI -> PT
for sentence in tqdm(test_tupi, desc="Tupi->PT"):
    translations2tupi.append(traduzir(sentence, TUPI, PORT, tokenizer, model))


Bidirecional PT↔Tupi: 100%|██████████| 1062/1062 [34:56<00:00,  1.97s/it] 


<h3> Limpeza do texto gerado </h3>

In [43]:
translations2pt_clean = [clean_pred_pt(t) for t in translations2pt]
translations2tupi_clean = [clean_pred_tupi(t) for t in translations2tupi]

In [ ]:
# Exportação para CSV (opcional)

import pandas as pd

df_out_tupi = pd.DataFrame({
    "portugues": test_pt,
    "tupi_pred": translations2tupi_clean
})

df_out_pt = pd.DataFrame({
    "tupi": test_tupi,
    "portugues_pred": translations2pt_clean
})

df_out_tupi.to_csv("pt2tupi_zeroShot.csv", index=False)
df_out_pt.to_csv("tupi2pt_zeroShot.csv", index=False)


Ajuste das linhas do CSV no formato "frase em pt","frase em tupi"

In [ ]:
fix_csv("pt2tupi_zeroShot.csv", "pt2tupi_zeroShot_fixed.csv")
fix_csv("tupi2pt_zeroShot.csv", "tupi2pt_zeroShot_fixed.csv")

Para não precisar traduzir denovo (caso já tenha os arquivos pt2tupi_zeroShot.csv e tupi2pt_zeroShot.csv)

In [23]:
dataf = pd.read_csv("pt2tupi_zeroShot_fixed.csv")
translations2tupi_clean = dataf['tupi_pred']

dataf = pd.read_csv("tupi2pt_zeroShot_fixed.csv")
translations2pt_clean = dataf['portugues_pred']

<h3>Avaliação Zero-Shot (BLEU, chrF1, chrF3)</h3>

In [25]:

# Avaliação
import evaluate

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# PT -> Tupi

refs_tupi_nested = [[r] for r in dataset["test"]["tupi"]]      
refs_tupi = dataset["test"]["tupi"]                            
preds_tupi = translations2tupi_clean

bleu_score = bleu.compute(predictions=preds_tupi, references=refs_tupi_nested)
chrf1 = chrf.compute(predictions=preds_tupi, references=refs_tupi, beta=1.0)
chrf3 = chrf.compute(predictions=preds_tupi, references=refs_tupi, beta=3.0)

with open("results_zero_shot.txt", "w", encoding="utf-8") as f:
    f.write("PT→Tupi:\n")
    f.write("BLEU: " + str(bleu_score['score']) + "\n")
    f.write(f"chrF1 " + str(chrf1["score"]) + '\n')
    f.write(f"chrF3 " + str(chrf3["score"]) + '\n')
    f.write("\n")

# Tupi -> PT

refs_pt_nested = [[r] for r in dataset["test"]["portugues"]] 
refs_pt = dataset["test"]["portugues"]                       
preds_pt = translations2pt_clean

bleu_score = bleu.compute(predictions=preds_pt, references=refs_pt_nested)
chrf1 = chrf.compute(predictions=preds_pt, references=refs_pt, beta=1.0)
chrf3 = chrf.compute(predictions=preds_pt, references=refs_pt, beta=3.0)

with open("results_zero_shot.txt", "a", encoding="utf-8") as f:
    f.write("PT→Tupi:\n")
    f.write("BLEU: " + str(bleu_score['score']) + "\n")
    f.write(f"chrF1 " + str(chrf1["score"]) + '\n')
    f.write(f"chrF3 " + str(chrf3["score"]) + '\n')

print("Arquivo results_zero_shot.txt gerado.")

Arquivo results_zero_shot.txt gerado.


<h3> Top 10 Melhores/Piores scores BLEU - Few_shot

In [26]:
from sacrebleu.metrics import BLEU

bleu_sentence = BLEU(effective_order=True)

sentence_scores_pt2tupi = []

for pred, ref in zip(preds_tupi, refs_tupi):
    score = bleu_sentence.sentence_score(pred, [ref]).score
    sentence_scores_pt2tupi.append({
        "portugues": ref,
        "tupi_pred": pred,
        "bleu": score
    })

# Ordenar
sentence_scores_pt2tupi_sorted = sorted(sentence_scores_pt2tupi, key=lambda x: x["bleu"], reverse=True)

top10 = sentence_scores_pt2tupi_sorted[:10]
bottom10 = sentence_scores_pt2tupi_sorted[-10:]

pd.DataFrame(top10).to_csv("pt2tupi_zeroShot_best10.csv", index=False, encoding="utf-8")
pd.DataFrame(bottom10).to_csv("pt2tupi_zeroShot_worst10.csv", index=False, encoding="utf-8")


sentence_scores_tupi2pt = []

for pred, ref in zip(preds_pt, refs_pt):
    score = bleu_sentence.sentence_score(pred, [ref]).score
    sentence_scores_tupi2pt.append({
        "tupi": ref,
        "portugues_pred": pred,
        "bleu": score
    })

sentence_scores_tupi2pt_sorted = sorted(sentence_scores_tupi2pt, key=lambda x: x["bleu"], reverse=True)

top10_pt = sentence_scores_tupi2pt_sorted[:10]
bottom10_pt = sentence_scores_tupi2pt_sorted[-10:]

pd.DataFrame(top10_pt).to_csv("tupi2pt_zeroShot_best10.csv", index=False, encoding="utf-8")
pd.DataFrame(bottom10_pt).to_csv("tupi2pt_zeroShot_worst10.csv", index=False, encoding="utf-8")

# Fine-tuning

In [5]:
tokenizer_pt2tupi = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    src_lang=PORT,
    tgt_lang=TUPI
)

tokenizer_tupi2pt = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    src_lang=TUPI,
    tgt_lang=PORT
)

# Função de pré-processamento
def preprocess_pt2tupi(examples):

    inputs = examples["portugues"]

    targets = examples["tupi"]

    model_inputs = tokenizer_pt2tupi(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer_pt2tupi(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_tupi2pt(examples):

    inputs = examples["tupi"]
    targets = examples["portugues"]

    model_inputs = tokenizer_tupi2pt(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer_tupi2pt(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_pt2tupi = { split: dataset[split].map(preprocess_pt2tupi, batched=True) for split in ["train", "validation"] }
tokenized_tupi2pt = { split: dataset[split].map(preprocess_tupi2pt, batched=True) for split in ["train", "validation"] }



Map:   0%|          | 0/4906 [00:00<?, ? examples/s]

Map:   0%|          | 0/1062 [00:00<?, ? examples/s]

Map:   0%|          | 0/4906 [00:00<?, ? examples/s]

Map:   0%|          | 0/1062 [00:00<?, ? examples/s]

<h3> Opcional utilizar peft para realizar LoRa
Utilizando a biblioteca [PEFT](https://huggingface.co/docs/peft/index)

In [7]:
model_pt2tupi = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT, device_map="cpu")
model_tupi2pt = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT, device_map="cpu")

In [8]:
from peft import LoraConfig, get_peft_model, TaskType
# 2. Define a configuração do LoRA
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # Aplica nas camadas de atenção
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# 3.Envelopa o modelo com PEFT
model_pt2tupi = get_peft_model(model_pt2tupi, lora_config)
model_tupi2pt = get_peft_model(model_tupi2pt, lora_config)
model_pt2tupi.print_trainable_parameters()
model_tupi2pt.print_trainable_parameters()
# Os parâmetros treináveis devem ser aproximadamente 0.5% ou menos

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821
trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


<h3> Configuração de avaliação do modelo
- Utilizando somente [BLEU](https://huggingface.co/spaces/evaluate-metric/bleu)
- Diferença entre sacrebleu e bleu [A Call for Clarity in Reporting BLEU Scores](https://arxiv.org/abs/1804.08771)

In [9]:
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")
def compute_metrics(tokenizer):
    def _fn(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_labels = [[l] for l in decoded_labels]
        bleu_result = metric.compute(predictions=decoded_preds, references=decoded_labels)
        return {"bleu": bleu_result["score"]}
    return _fn

In [ ]:

collator_pt2tupi = DataCollatorForSeq2Seq(
    tokenizer_pt2tupi,
    model=model_pt2tupi,
    label_pad_token_id=-100
)

collator_tupi2pt = DataCollatorForSeq2Seq(
    tokenizer_tupi2pt,
    model=model_tupi2pt,
    label_pad_token_id=-100
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-finetuned",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs = 5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=None,
    fp16=False,
    bf16=False,
    predict_with_generate=True, 
    use_cpu=True,
    #metric_for_best_model="bleu",    # Deveria ter sido usado no treinamento
    #greater_is_better=True,          # Deveria ter sido usado no treinamento
    load_best_model_at_end=True
)


trainer_pt2tupi = Seq2SeqTrainer(
    model=model_pt2tupi,
    args=training_args,  
    train_dataset=tokenized_pt2tupi["train"],
    eval_dataset=tokenized_pt2tupi["validation"],
    data_collator=collator_pt2tupi,
    processing_class=tokenizer_pt2tupi,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=compute_metrics(tokenizer_pt2tupi)
)

output_pt2tupi = trainer_pt2tupi.train()
trainer_pt2tupi.save_model("./nllb-ft-pt2tupi")   # salva adapter/estado
print("Fase1 metrics:", output_pt2tupi.metrics)

trainer_tupi2pt = Seq2SeqTrainer(
    model=model_tupi2pt,
    args=training_args,
    train_dataset=tokenized_tupi2pt["train"],
    eval_dataset=tokenized_tupi2pt["validation"],
    data_collator=collator_tupi2pt,
    processing_class=tokenizer_tupi2pt,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=compute_metrics(tokenizer_tupi2pt),
)

output_tupi2pt = trainer_tupi2pt.train()
trainer_tupi2pt.save_model("./nllb-ft-tupi2pt")   # salva adapter/estado
print("Fase2 metrics:", output_tupi2pt.metrics)


The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss,Bleu
1,4.647800,4.233756,0.355717
2,4.291000,3.935557,0.470439
3,4.040600,3.795506,0.446001
4,3.979800,3.720820,0.454227
5,3.912500,3.695590,0.467424


/home/vitorvg/miniforge3/envs/nlp/lib/python3.10/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce MX350 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/vitorvg/miniforge3/envs/nlp/lib/python3.10/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/vitorvg/miniforge3/envs/nlp/lib/python3.10/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce MX350 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce MX350 GPU with PyTorch, please check the in

Fase1 metrics: {'train_runtime': 29894.8913, 'train_samples_per_second': 0.821, 'train_steps_per_second': 0.205, 'total_flos': 919800105467904.0, 'train_loss': 4.221503824670818, 'epoch': 5.0}


Epoch,Training Loss,Validation Loss,Bleu
1,3.562400,3.290540,0.659844
2,3.417500,3.165364,0.988228
3,3.297200,3.092322,0.748181
4,3.203600,3.055538,0.803171
5,3.211100,3.041938,0.796633


Fase2 metrics: {'train_runtime': 15548.8207, 'train_samples_per_second': 1.578, 'train_steps_per_second': 0.395, 'total_flos': 1153056619511808.0, 'train_loss': 3.3524515887247226, 'epoch': 5.0}


In [ ]:

# Traduções com o modelo fine-tunado
checkpoint_path_pt2tupi = "nllb-ft-pt2tupi"

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path_pt2tupi)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path_pt2tupi)

tokenizer.src_lang = PORT
tokenizer.tgt_lang = TUPI

from tqdm import tqdm

test_pt = dataset["test"]["portugues"]     # frases de entrada em pt
test_tupi = dataset["test"]["tupi"]        # frases de entrada em tupi
translations2tupi_ft = []                  # armazenará as traduções em tupi
translations2pt_ft = []                    # armazenará as traduções em pt

# PT -> TUPI
for sentence in tqdm(test_pt, desc="PT->Tupi"):
    translations2tupi_ft.append(traduzir(sentence, PORT, TUPI, tokenizer, model))

checkpoint_path_tupi2pt = "nllb-ft-tupi2pt"

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path_tupi2pt)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path_tupi2pt)

tokenizer.src_lang = TUPI
tokenizer.tgt_lang = PORT

# TUPI -> PT
for sentence in tqdm(test_tupi, desc="Tupi->PT"):
    translations2pt_ft.append(traduzir(sentence, TUPI, PORT, tokenizer, model))

# Limpeza
translations2tupi_ft_clean = [clean_pred_tupi(t) for t in translations2tupi_ft]
translations2pt_ft_clean = [clean_pred_pt(t) for t in translations2pt_ft]


# Exportação para CSV (opcional)

import pandas as pd

df_out_tupi = pd.DataFrame({
    "portugues": test_pt,
    "tupi_pred": translations2tupi_ft_clean
})

df_out_pt = pd.DataFrame({
    "tupi": test_tupi,
    "portugues_pred": translations2pt_ft_clean
})

df_out_tupi.to_csv("pt2tupi_fewShot.csv", index=False)
df_out_pt.to_csv("tupi2pt_fewShot.csv", index=False)


Tupi->PT: 100%|██████████| 1062/1062 [1:16:33<00:00,  4.33s/it] 


TypeError: TextIOWrapper.write() takes exactly one argument (2 given)

Ajusta linha do CSV no formato "frase em pt","frase em tupi"

In [19]:
fix_csv("pt2tupi_fewShot.csv", "pt2tupi_fewShot_fixed.csv")
fix_csv("tupi2pt_fewShot.csv", "tupi2pt_fewShot_fixed.csv")

Arquivo corrigido salvo em pt2tupi_fewShot_fixed.csv
Arquivo corrigido salvo em tupi2pt_fewShot_fixed.csv


In [ ]:

# Avaliação
import evaluate

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

# PT -> Tupi

refs_tupi_nested = [[r] for r in dataset["test"]["tupi"]]      
refs_tupi = dataset["test"]["tupi"]                            
preds_tupi = translations2tupi_ft_clean

bleu_score = bleu.compute(predictions=preds_tupi, references=refs_tupi_nested)
chrf1 = chrf.compute(predictions=preds_tupi, references=refs_tupi, beta=1.0)
chrf3 = chrf.compute(predictions=preds_tupi, references=refs_tupi, beta=3.0)

with open("results_few_shot.txt", "w", encoding="utf-8") as f:
    f.write("PT→Tupi:\n")
    f.write("BLEU: " + str(bleu_score['score']) + "\n")
    f.write(f"chrF1 " + str(chrf1["score"]) + '\n')
    f.write(f"chrF3 " + str(chrf3["score"]) + '\n')
    f.write("\n")

# Tupi -> PT

refs_pt_nested = [[r] for r in dataset["test"]["portugues"]] 
refs_pt = dataset["test"]["portugues"]                       
preds_pt = translations2pt_ft_clean

bleu_score = bleu.compute(predictions=preds_pt, references=refs_pt_nested)
chrf1 = chrf.compute(predictions=preds_pt, references=refs_pt, beta=1.0)
chrf3 = chrf.compute(predictions=preds_pt, references=refs_pt, beta=3.0)

with open("results_few_shot.txt", "a", encoding="utf-8") as f:
    f.write("PT→Tupi:\n")
    f.write("BLEU: " + str(bleu_score['score']) + "\n")
    f.write(f"chrF1 " + str(chrf1["score"]) + '\n')
    f.write(f"chrF3 " + str(chrf3["score"]) + '\n')

# Número de teraflops utilizados
# Número total operaçõeS de número de ponto-flutuante (total_flos)
with open("results_few_shot.txt", "a", encoding="utf-8") as f:
    f.write("Para Modelos que minimizam a Loss:\n")
    f.write("PT -> TUPI:\n")
    f.write(str(output_pt2tupi) + '\n')
    f.write(str(output_pt2tupi.metrics["total_flos"] / (output_pt2tupi.metrics["train_runtime"] *  1e12)) + '\n')
    f.write("TUPI -> PT:\n")
    f.write(str(output_tupi2pt) + '\n')
    f.write(str(output_tupi2pt.metrics["total_flos"] / (output_tupi2pt.metrics["train_runtime"] *  1e12)) + '\n')

print("Arquivo results_few_shot.txt gerado.")


Arquivo results_few_shot.txt gerado.


<h3> Top 10 Melhores/Piores scores BLEU - Few_shot

In [21]:
from sacrebleu.metrics import BLEU

bleu_sentence = BLEU(effective_order=True)

sentence_scores_pt2tupi = []

for pred, ref in zip(preds_tupi, refs_tupi):
    score = bleu_sentence.sentence_score(pred, [ref]).score
    sentence_scores_pt2tupi.append({
        "portugues": ref,
        "tupi_pred": pred,
        "bleu": score
    })

# Ordenar
sentence_scores_pt2tupi_sorted = sorted(sentence_scores_pt2tupi, key=lambda x: x["bleu"], reverse=True)

top10 = sentence_scores_pt2tupi_sorted[:10]
bottom10 = sentence_scores_pt2tupi_sorted[-10:]

pd.DataFrame(top10).to_csv("pt2tupi_fewShot_best10.csv", index=False, encoding="utf-8")
pd.DataFrame(bottom10).to_csv("pt2tupi_fewShot_worst10.csv", index=False, encoding="utf-8")


sentence_scores_tupi2pt = []

for pred, ref in zip(preds_pt, refs_pt):
    score = bleu_sentence.sentence_score(pred, [ref]).score
    sentence_scores_tupi2pt.append({
        "tupi": ref,
        "portugues_pred": pred,
        "bleu": score
    })

sentence_scores_tupi2pt_sorted = sorted(sentence_scores_tupi2pt, key=lambda x: x["bleu"], reverse=True)

top10_pt = sentence_scores_tupi2pt_sorted[:10]
bottom10_pt = sentence_scores_tupi2pt_sorted[-10:]

pd.DataFrame(top10_pt).to_csv("tupi2pt_fewShot_best10.csv", index=False, encoding="utf-8")
pd.DataFrame(bottom10_pt).to_csv("tupi2pt_fewShot_worst10.csv", index=False, encoding="utf-8")
